<a href="https://colab.research.google.com/github/deburky/language-models/blob/main/gpt-oss-claude-code/colab-eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Evaluation: gpt-oss-claude-code

Evaluates `deburky/gpt-oss-claude-code` on knowledge Q&A and tool-calling tasks.  
Produces a markdown report saved to `eval_report.md`.

Authored by: [Denis Burakov](https://www.github.com/deburky)

## 1. Install dependencies

In [1]:
%pip install transformers torch datasets -q

## 2. Load model

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'deburky/gpt-oss-claude-code'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
print('Model loaded.')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/391 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

Model loaded.


## 3. Helpers

In [3]:
import re
import json
import time
import pathlib
import glob as globlib
from datetime import datetime

TOOL_SYSTEM = """You are a coding assistant with access to the file system.

Available tools:
- read(path, offset, limit): Read a file. Use limit to cap lines returned.
- glob(pat): Find files by pattern
- grep(pat, path): Search for text in files

To use a tool, format it EXACTLY like this:
<tool_call>{"tool": "name", "args": {"key": "value"}}</tool_call>

When asked to read the first N lines, always pass limit: N. Reply concisely."""

KNOWLEDGE_SYSTEM = 'You are a knowledgeable assistant. Answer concisely and accurately.'

KNOWLEDGE_PROMPTS = [
    'Who invented the method of Maximum Likelihood Estimation?',
    'Who is Alan Turing and what is he famous for?',
    'What is Weight of Evidence (WoE) in credit scoring and how is it calculated?',
    'Explain the difference between L1 and L2 regularization.',
    'What is the purpose of a chat template in large language models?',
]

TOOL_PROMPTS = [
    'List all Python files in the current directory.',
    'Read the first 20 lines of /etc/hostname.',
    'Search for the word root in /etc/hostname.',
]


def generate(messages, max_new_tokens=512):
    t0 = time.perf_counter()
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors='pt', return_dict=True,
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.3, do_sample=True)
    elapsed = time.perf_counter() - t0
    raw = tokenizer.decode(output_ids[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=False)
    if '<|channel|>final<|message|>' in raw:
        raw = raw.split('<|channel|>final<|message|>')[-1]
    raw = re.sub(r'<\|[^>]+\|>', '', raw).strip()
    return raw, elapsed


def run_tool(name, args):
    if name == 'glob':
        files = globlib.glob(args.get('pat', '*'), recursive=True)
        return '\n'.join(sorted(files)) or 'none'
    if name == 'read':
        p = pathlib.Path(args['path'])
        lines = p.read_text(errors='replace').splitlines()
        offset = int(args.get('offset', 0))
        limit = int(args.get('limit', 20))
        return ''.join(f"{offset+i+1:4}| {l}\n" for i, l in enumerate(lines[offset:offset+limit]))
    if name == 'grep':
        pattern = re.compile(args.get('pat', ''))
        p = pathlib.Path(args.get('path', '.'))
        hits = []
        for fp in ([p] if p.is_file() else p.rglob('*')):
            if fp.is_file():
                for i, line in enumerate(fp.read_text(errors='replace').splitlines(), 1):
                    if pattern.search(line):
                        hits.append(f'{fp}:{i}:{line.rstrip()}')
        return '\n'.join(hits[:40]) or 'none'
    return f'error: unknown tool {name}'


def parse_calls(text):
    calls = []
    for tag in ['glob', 'read', 'grep']:
        for m in re.finditer(rf'<{tag}>(.*?)</{tag}>', text, re.DOTALL):
            try:
                calls.append({'name': tag, 'args': json.loads(m.group(1))})
            except Exception:
                pass
    for m in re.finditer(r'<tool_call>(.*?)</tool_call>', text, re.DOTALL):
        try:
            d = json.loads(m.group(1))
            if d.get('tool') in ('glob', 'read', 'grep'):
                calls.append({'name': d['tool'], 'args': d.get('args', {})})
        except Exception:
            pass
    return calls


def parse_raw_to_message(raw):
    thinking, content = '', ''
    if '<|channel|>analysis<|message|>' in raw:
        part = raw.split('<|channel|>analysis<|message|>')[1]
        thinking = part.split('<|end|>')[0].strip()
    if '<|channel|>final<|message|>' in raw:
        part = raw.split('<|channel|>final<|message|>')[1]
        content = re.split(r'<\|end\|>|<\|return\|>', part)[0].strip()
    else:
        content = re.sub(r'<\|[^>]+\|>', '', raw).strip()
    msg = {'role': 'assistant', 'content': content}
    if thinking:
        msg['thinking'] = thinking
    return msg


def run_knowledge(prompt):
    messages = [
        {'role': 'system', 'content': KNOWLEDGE_SYSTEM},
        {'role': 'user', 'content': prompt},
    ]
    return generate(messages, max_new_tokens=256)


def run_tool_task(task, max_steps=5):
    messages = [
        {'role': 'system', 'content': TOOL_SYSTEM},
        {'role': 'user', 'content': task},
    ]
    tool_log = []
    total_time = 0.0
    for _ in range(max_steps):
        response, elapsed = generate(messages, max_new_tokens=512)
        total_time += elapsed
        calls = parse_calls(response)
        tool_log.append({'response': response, 'tool_calls': []})
        if not calls:
            break
        for call in calls:
            result = run_tool(call['name'], call['args'])
            tool_log[-1]['tool_calls'].append({'tool': call['name'], 'args': call['args'], 'result': result[:400]})
            messages.append(parse_raw_to_message(response))
            messages.append({'role': 'user', 'content': f"Tool {call['name']} returned:\n{result}"})
    return tool_log, total_time

print('Helpers ready.')

Helpers ready.


## 4. Run evaluation

In [4]:
knowledge_results = []
tool_results = []

print('[Knowledge Q&A]')
for prompt in KNOWLEDGE_PROMPTS:
    print(f'Q: {prompt[:60]}...')
    answer, latency = run_knowledge(prompt)
    knowledge_results.append({'prompt': prompt, 'answer': answer, 'latency': latency})
    print(f'{latency:.2f}s — {answer[:80]}')

print('\n[Tool Calling]')
for task in TOOL_PROMPTS:
    print(f'T: {task}')
    log, latency = run_tool_task(task)
    tool_results.append({'task': task, 'log': log, 'latency': latency})
    print(f'{latency:.2f}s — {len(log)} step(s)')

print('\nDone.')

[Knowledge Q&A]
  Q: Who invented the method of Maximum Likelihood Estimation?...
  101.61s — Maximum Likelihood Estimation was formalized by Ronald A. Fisher in 1922. It is 
  Q: Who is Alan Turing and what is he famous for?...
  144.68s — Alan Turing (1912–1954) was a British mathematician and logician. He is famous f
  Q: What is Weight of Evidence (WoE) in credit scoring and how i...
  237.02s — Weight of Evidence (WoE) is a transformation used in credit scoring to convert c
  Q: Explain the difference between L1 and L2 regularization....
  113.08s — L1 adds the absolute values of weights (sum |w|), encouraging sparsity (many wei
  Q: What is the purpose of a chat template in large language mod...
  91.20s — A chat template is a structured prompt that defines the roles (e.g., user, syste

[Tool Calling]
  T: List all Python files in the current directory.
  71.87s — 2 step(s)
  T: Read the first 20 lines of /etc/hostname.
  1057.27s — 3 step(s)
  T: Search for the word root in /etc

## 5. Build and save report

In [5]:
def fmt_tool_log(log):
    lines = []
    for step in log:
        if step['response']:
            lines.append(step['response'])
        for tc in step['tool_calls']:
            lines.append(f"\n**→ {tc['tool']}({json.dumps(tc['args'])})**")
            lines.append(f"```\n{tc['result']}\n```")
    return '\n'.join(lines).strip()


now = datetime.now().strftime('%Y-%m-%d %H:%M')
lines = [
    '# Evaluation Report',
    '',
    f'**Model:** `{MODEL_ID}`  ',
    f'**Date:** {now}',
    '',
    '---',
    '',
    '## 1. Knowledge Q&A',
    '',
    '| Question | Answer | Latency |',
    '|---|---|---|',
]
for r in knowledge_results:
    ans = r['answer'].replace('\n', '<br>').replace('|', '\\|')
    lines.append(f"| {r['prompt']} | {ans} | {r['latency']:.2f}s |")

lines += ['', '---', '', '## 2. Tool Calling', '']
for r in tool_results:
    lines += [
        f"### {r['task']}",
        '',
        f"**Steps:** {len(r['log'])}  **Latency:** {r['latency']:.2f}s",
        '',
        fmt_tool_log(r['log']),
        '',
    ]

report = '\n'.join(lines)
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
out_path = f'eval_{ts}.md'
with open(out_path, 'w') as f:
    f.write(report)
print(f'Report saved: {out_path}')
print(report[:500])

Report saved: eval_20260403_162210.md
# Evaluation Report

**Model:** `deburky/gpt-oss-claude-code`  
**Date:** 2026-04-03 16:22

---

## 1. Knowledge Q&A

| Question | Answer | Latency |
|---|---|---|
| Who invented the method of Maximum Likelihood Estimation? | Maximum Likelihood Estimation was formalized by Ronald A. Fisher in 1922. It is a general method that has been used by many statisticians and scientists since then. | 101.61s |
| Who is Alan Turing and what is he famous for? | Alan Turing (1912–1954) was a British mathemati
